# Exam 1 Grading

This notebook grades Exam 1 using regradex. The workflow is:

1. Load student responses and grading config
2. Sanity check — verify correct answers match their own patterns
3. Autograde all questions
4. Manual review — inspect and score unrecognized answers per question
5. Student lookup — review any individual student's results
6. Score review — summary stats and flag low-scoring questions
7. Save final results

**Data files** (gitignored, not committed):
- `data/Exam1.xlsx` — raw student responses from LMS
- `data/Exam1_graded.xlsx` — output with regradex scores
- `json/answers.json` — grading config (run `json/make_answers.py` to regenerate)

## Setup

In [1]:
import json
import numpy as np
import pandas as pd

from regradex import (
    validate_all,
    compile_question,
    autograde_all,
    review_queue,
    apply_manual,
    final_grade,
    grade_all,
)

# Grading curve — a student scoring 200/200 gets 1.0, 
# each point below 200 costs SCALE_FACTOR/200 of their grade
SCALE_FACTOR: float = 0.5
TOTAL_POINTS: int = 200

In [2]:
# Load student responses
df = pd.read_excel('data/Exam1.xlsx')
print(f'Loaded {len(df)} students, {len(df.columns)} columns')

Loaded 24 students, 220 columns


In [3]:
# Load, validate, and compile grading config
with open('json/answers.json', 'r') as f:
    configs = json.load(f)

validate_all(configs)
print('Validation passed')

questions = {
    key: compile_question(config)
    for key, config in configs.items()
    if 'regex' in config
}
print(f'Compiled {len(questions)} autograded questions')
print(f'Manual only: {len(configs) - len(questions)} questions')

Validation passed
Compiled 14 autograded questions
Manual only: 6 questions


## Sanity check

Verify that each question's canonical correct answer matches its own patterns.
Any `False` values mean the pattern doesn't catch the answer you wrote — fix the pattern before grading.

In [4]:
import re

all_ok = True
for key, question in questions.items():
    answer = configs[key]['answer']
    if isinstance(answer, dict):
        # Question 31 has model-specific answers — check each one
        for model, model_answer in answer.items():
            matches = [bool(p.search(str(model_answer))) for p in question.patterns]
            if not all(matches):
                print(f'Q{key} ({model}): {matches}')
                all_ok = False
    else:
        matches = [bool(p.search(str(answer))) for p in question.patterns]
        if not all(matches):
            print(f'Q{key}: {matches}')
            all_ok = False

if all_ok:
    print('All correct answers match their patterns')

Q10: [True, False, True]
Q11: [True, False, True]


## Autograde

Run the grading engine over all student responses. This adds `rg_auto_N` columns for each autograded question.

In [5]:
df = autograde_all(df, questions)
print('Autograding complete')

# Quick summary — mean auto score per question
auto_cols = {key: f'rg_auto_{key}' for key in questions}
means = {key: df[col].mean() for key, col in auto_cols.items()}
for key, mean in means.items():
    queue = review_queue(df, key)
    print(f'Q{key}: mean={mean:.1f}, needs review={len(queue)}')

Autograding complete
Q6: mean=4.6, needs review=13
Q9: mean=7.7, needs review=1
Q10: mean=7.7, needs review=0
Q11: mean=7.5, needs review=1
Q13: mean=5.0, needs review=12
Q15: mean=8.3, needs review=4
Q17: mean=3.8, needs review=11
Q20: mean=4.6, needs review=9
Q21: mean=5.8, needs review=10
Q22: mean=5.0, needs review=12
Q29: mean=5.4, needs review=11
Q31: mean=2.9, needs review=16
Q35: mean=7.9, needs review=5
Q36: mean=7.5, needs review=6


## Manual review

For each question, inspect unrecognized answers and apply manual scores.
Run the question cell, then use `apply_manual()` to record scores.

Scores will be saved to `rg_manual_N` columns. Manual scores take precedence
over auto scores in the final grade.

In [6]:
def show_queue(question_key: str) -> None:
    """Print the manual review queue for a question."""
    question = questions[question_key]
    config = configs[question_key]
    queue = review_queue(df, question_key)

    print(f'Question {question_key}: {config["question"]}')
    print(f'Correct answer: {config["answer"]}')
    print(f'Needs review: {len(queue)}\n')

    for idx, answer in queue.items():
        student = df.loc[idx, 'Username']
        print(f'  [{idx}] {student}: {answer}')

In [7]:
# Review question 6
show_queue('6')

Question 6: What's the average wage in the sample? What's the standard deviation of the wage?
Correct answer: $1532700/year, $996567.1/year
Needs review: 13

  [0] znk150030: The average wage is 1324.224\n<br>\nThe standard deviation of the wage is 195.643
  [2] rxa170002: Average wage in the sample: 1532.7\nStandard deviation of the wage: 996.5671
  [4] hxk190028: Average wage is 1532.7\nStandard deviation of wage is 996.5671
  [5] mem200007: The average wage is 1532.7 and the standard deviation is 996.5671.
  [6] hxk210027: The average wage is 1532.7, and the standard deviation of the wage is 996.5671
  [8] rxs210093: 1265.5 and 996.57
  [9] cxq210001: the average wage in the sample is 1532.7.\nthe standard deviation of the wage is 996.5671
  [10] nxu210006: Using wpull function  we load data from sql to R  and nbasa1 is extracted alon g with wage.\nThe standard deviation for wage would be $996.5671 per yearand  and from the output above the average wage would be (mean in sample )wou

In [ ]:
# Apply manual scores for question 6
# df = apply_manual(df, '6', index, score)
# Example:
# df = apply_manual(df, '6', 12, 5)

In [8]:
# Review question 9
show_queue('9')

Question 9: Interpret the estimated coefficient on age in modela.
Correct answer: 1 year increase in age ==> $91300/year in wage
Needs review: 1

  [16] nic220000: For every additional unit increase in age, there is a predicted increase of $9130 in the annual salary. 


In [9]:
# Apply manual scores for question 9
# df = apply_manual(df, '9', index, score)

In [10]:
# Review question 10
show_queue('10')

Question 10: Using modela, predict the estimated wage for a player who is 27 years old, has a draft number=7, and scores 5 points per game.
Correct answer: $1117662/year
Needs review: 0



In [11]:
# Apply manual scores for question 10
# df = apply_manual(df, '10', index, score)

In [12]:
# Review question 11
show_queue('11')

Question 11: Using modela, predict the estimated wage for a player who is 27 years old, has a draft number=28, and scores 14 points per game.
Correct answer: $1686348/year
Needs review: 1

  [4] hxk190028: 686.9


In [13]:
# Apply manual scores for question 11
# df = apply_manual(df, '11', index, score)

In [14]:
# Review question 13
show_queue('13')

Question 13: What complicated hypothesis is being tested in modelc?
Correct answer: b_4=b_5 or the coefficient on assists is equal the coefficient on rebounds
Needs review: 12

  [0] znk150030: There is no significant variation towards wages when rebounds and assists are combined into a singular variable.
  [3] cll180003: Whether Rebounds and Assists combined are jointly significant factors in influencing wage
  [4] hxk190028: the hypothesis being tested in this model is whether the combination of the predictor variables can significantly explain the variation in wage. \nthe model test whter there is a linear relationship between wage and each of the predictor variables while allowing for a potential interation between rebounds and assists.
  [5] mem200007: The inclusion of I(rebound+assists) indicates that the hypothesis is testing if there is an combined effect of rebounds and assists on a player's wage over these specifc variables seperately. In other words, its testing whether the 

In [15]:
# Apply manual scores for question 13
# df = apply_manual(df, '13', index, score)

In [16]:
# Review question 15
show_queue('15')

Question 15: What hypothesis is being tested in the anova with modeld?
Correct answer: b_4=b_5=0 or joint significance of rebounds and assists
Needs review: 4

  [0] znk150030: There is no sginificant difference between model b and d.
  [9] cxq210001: The null hypothesis being tested is that the two models (modelb and modeld) are equivalent, and whether there is a significant difference in fit between these two models.\nNull hypothesis: modelb does not provide a significantly better fit than modeld.\nAlternative hypothesis: the additional terms in modelb significantly contribute to explain the variance in the dependent variable (wage)
  [13] aer220002: We are testing the variances of model d to model b and determining if the variances are the same for each group or different.
  [18] vxg230021: The hypothesis is being tested in the anova with modeld is if the extra variables ( contribute and assists) contribute to explaining the variance in wage.


In [17]:
# Apply manual scores for question 15
# df = apply_manual(df, '15', index, score)

In [18]:
# Review question 17
show_queue('17')

Question 17: In modeld, what's a 95% confidence interval for a player who is 27 years old, plays 1800 minutes per year, scores 11 points per game, and has a draft number=20?
Correct answer: $1421000/year to $1597000/year
Needs review: 11

  [0] znk150030: 95% confidence interval would be between 74.346 and 90.854
  [3] cll180003: LCL: 1524.84; UCL: 1848.18
  [4] hxk190028: 1.970+27age+1800minutes+11point+20draft
  [5] mem200007: the confidence interval is 1509
  [7] mxm210122: <Unanswered>
  [8] rxs210093: <Unanswered>
  [10] nxu210006: wage =1509+91.9*I(age-27)-0.0520*I(minutes-1800)+97.6*I(points-11)-11.9*I(draft-20)\n95% confidence interval for 27 year old player who plays 1800 minutes per year and 11 point score per game and draft no=20 since all the indicators  equal to  0 contirbution for the wage beyond the intercept in model .
  [11] hxf220001: confidence interval = (1016.9 -  201.736 , 1016.9 + 201.736 ) = ( 815.164, 1218.636 ) 
  [13] aer220002: (103.84, 251.176)
  [17] txy23

In [19]:
# Apply manual scores for question 17
# df = apply_manual(df, '17', index, score)

In [20]:
# Review question 20
show_queue('20')

Question 20: Using modelf, if the draft number increases by 1 position, what effect will that have on wage if draft=10?
Correct answer: -$27909/year
Needs review: 9

  [0] znk150030: If the draft number increases by 1 position, from 10 to 11, the wage is expected to decrease by $33.1, assumming all other variables are constant. 
  [2] rxa170002:  if the draft number increases by 1 position, wage will decrease by $-695.1 if draft=10\ncalculation:\n-33.1+ 2* (-33.1) *10
  [4] hxk190028: there will be no effect on the wage
  [7] mxm210122: <Unanswered>
  [10] nxu210006: if draft number is increased by 1 position which means drafting later and is considered less favourable the wage would be decreased by draft coefficient.\nthe draft coefficient from model is -33.1 so it shows increase in draftnumber is associated with decrease in wage by $33.1 thousand where all other variables are constant.\nso if players draft number is 10 and it increases by 1 to 11 their wage is expected to decrease by

In [21]:
# Apply manual scores for question 20
# df = apply_manual(df, '20', index, score)

In [22]:
# Review question 21
show_queue('21')

Question 21: Interpret the estimated coefficient on age in modelg.
Correct answer: 1% increase in age ==> 1.77% increase in wage
Needs review: 10

  [0] znk150030: As age increases, wages tend to increase through a log method. The effect of age on wages may be more significant on older individuals, thought there may be more factors that are not yet observed in relation to the age coefficient estimate.
  [4] hxk190028: 1.77
  [8] rxs210093: 0.566
  [9] cxq210001: For a 10% increase on age for a basketball player, there will be 17.7% increase in the wage.
  [17] txy230005: After controling other variables, the average effect of age on annual wage is exp(1.77)
  [19] mxa230073: A 100% change in age will generate a 177% increase in wage.
  [20] bxs230029: The wage will increase by (1.77*100) = 177% if age increases by 100%.
  [21] pxs230061: When age increases by 100%, the wage increases by 177%.
  [22] kxm230072: with increase in 100% of age the wage increases 177%.
  [23] txb230031: a 10

In [23]:
# Apply manual scores for question 21
# df = apply_manual(df, '21', index, score)

In [24]:
# Review question 22
show_queue('22')

Question 22: Interpret the estimated coefficient on minutes in modelg.
Correct answer: 1 more minute ==> 0.00637% increase in wage
Needs review: 12

  [0] znk150030: The estimated effect of minutes played on the log of wages, for each additional minute plkayed, the log of wages is estimated to increase by $0.0000657 USD
  [4] hxk190028: 6.37e-05
  [5] mem200007: The estimated coefficient on minutes is 0.0000637 with a p-value of 4.16e-1. This suggests that a one unit increase in minutes is associated with a 0.0000637 increase in wage. However, since the p value is not statistically signnificant, it indicates that there is not enough statistical evidence to support this relationship, suggesting that minutes played is not a reliable predictor of wage in this model. 
  [8] rxs210093: 6.4e-05
  [9] cxq210001: For a 100 units increase on minutes for a basketball player, there will be 0.637% increase in the wage.
  [10] nxu210006: the coefficient of minutes is -0.000637 it is not log transfo

In [25]:
# Apply manual scores for question 22
# df = apply_manual(df, '22', index, score)

In [26]:
# Review question 29
show_queue('29')

Question 29: Interpret the estimated coefficient on CPI in modela. Does the model make sense?
Correct answer: 1% increase in cpi ==> 0.0822% increase in unemployment. No. Spurious
Needs review: 11

  [0] znk150030: The estimated coefficient on CPI indicates how much the expected value of the log of unemployement changes for a .0822 unit increase. This model lacks necessary variables to make sense and is limited based on what is being analyzed here. 
  [4] hxk190028: 0.0822
  [8] rxs210093: yes totally make sense difference between consecutive elements in a vector of time series is applied on the other models and modles forumla will be changed 
  [9] cxq210001: For a 0.1% increase in CPI, there is a 0.00822% increase in the unemployment. The model does make sense.
  [12] sre220000: The coefficient is not significant even at the 10% level. Since this is a time series data set, it seems like we may be having some issues with autocorrelation between the two. They may both be non-stationary

In [27]:
# Apply manual scores for question 29
# df = apply_manual(df, '29', index, score)

In [28]:
# Review question 31
show_queue('31')

Question 31: Interpret the estimated coefficient on CPI in the model you have selected.
Correct answer: {'modeld': '1% increase in cpi ==> 0.002% decrease in unemployment. Yes.', 'modele': '1% increase in cpi ==> 2.079% decrease in unemployment. Yes.', 'modelf': '1% increase in cpi ==> 0.801% decrease in unemployment. Yes.', 'modelh': '1% increase in cpi ==> 0.682% decrease in unemployment. Yes.', 'modeli': '1% increase in cpi ==> 3.018% decrease in unemployment. Yes.'}
Needs review: 16

  [0] znk150030: The estimated coefficient of -2.08 indicates that a one unit increase in the difference between the log of CPI associated with a decrease of 2.08 units in the difference with the log of unemployment
  [1] sxs150230: A one-unit increase in the difference between consecutive terms in the log(cpi) series is associated with a 2.08 decrease in the difference between consecutive terms in the log(unem) series.
  [2] rxa170002: Coefficient of CPI in modela is 0.0822\n1% increase in CPI is asso

In [29]:
# Apply manual scores for question 31
# df = apply_manual(df, '31', index, score)

In [30]:
# Review question 35
show_queue('35')

Question 35: When is unemployment stationary?
Correct answer: 1 difference, level
Needs review: 5

  [0] znk150030: Unemployment is stationary at KPSS level 0.0071196
  [4] hxk190028: Whent the KPSS Level is 0.063979<br>so at 2
  [8] rxs210093: 3.34
  [9] cxq210001: A stationary time series is one that has a constant mean and variance over time. The unemployment rate has a constant mean and variance over time. if the p-value is less than the significance level (e.g. 0.1), then we reject the null hypothesis and can conclude that the time series is not stationary. So, when the p-value is greater than the printed p-value (0.1), that indicates unemployment stationary.
  [16] nic220000: Yes, the unemployment variable is both level stationary and trend stationary with no differencing. Both p-values for the level stationary test and trend stationary test are below the 5% significance level. 


In [31]:
# Apply manual scores for question 35
# df = apply_manual(df, '35', index, score)

In [32]:
# Review question 36
show_queue('36')

Question 36: When is the CPI stationary?
Correct answer: 2 differences, level
Needs review: 6

  [0] znk150030: Unemployment is stationary at KPSS level 0.0071196
  [4] hxk190028: At 1, so when the KPSS Level is 13.031\nKPSS Trend is 1.8859
  [8] rxs210093: 6.592
  [9] cxq210001: A stationary time series is one that has a constant mean and variance over time. The CPI has a constant mean and variance over time. if the p-value is less than the significance level (e.g. 0.1), then we reject the null hypothesis and can conclude that the time series is not stationary. So, when the p-value is greater than the printed p-value (0.1), that indicates CPI stationary.
  [10] nxu210006: The CPI is stationary after the differencing twice  with p value 0.1.
  [16] nic220000: Yes, the CPI variable is both level stationary and trend stationary with no differencing and with one differencing taken. 


In [33]:
# Apply manual scores for question 36
# df = apply_manual(df, '36', index, score)

## Compute final grades

Manual scores take precedence over auto scores where present.
Adds `rg_grade_N` columns and an `rg_total` column.

In [34]:
df = grade_all(df, list(questions.keys()))

# Apply curve
df['rg_curved'] = 1 - SCALE_FACTOR * (TOTAL_POINTS - df['rg_total']) / TOTAL_POINTS

print(df[['rg_total', 'rg_curved']].describe().round(3))

       rg_total  rg_curved
count    24.000     24.000
mean     83.750      0.709
std      33.695      0.084
min      15.000      0.538
25%      73.750      0.684
50%      87.500      0.719
75%     106.250      0.766
max     140.000      0.850


## Student lookup

Look up an individual student's results by username.

In [35]:
netid = ''  # set netid here

row = df[df['Username'] == netid]

if row.empty:
    print(f'Student {netid} not found')
else:
    grade_cols = [f'rg_grade_{key}' for key in questions]
    wrong = [col for col in grade_cols if row[col].item() < 10]

    print(f'Student: {netid}')
    print(f'Total: {row["rg_total"].item()}')
    print(f'Curved: {row["rg_curved"].item():.3f}')
    percentile = np.where(df['rg_curved'] < row['rg_curved'].item(), 1, 0).mean()
    print(f'Percentile: {percentile:.1%}')
    print()

    for col in wrong:
        key = col[9:]  # strip 'rg_grade_'
        config = configs[key]
        print(f'Q{key}: {config["question"]}')
        print(f'  Correct answer: {config["answer"]}')
        print(f'  Student answer: {row["Answer " + key].item()}')
        print(f'  Score: {row[col].item()}/10')
        print()

Student  not found


## Score review

Summary statistics and questions with low mean scores — worth a second look.

In [36]:
# Overall distribution
df[['rg_total', 'rg_curved']].describe().round(3)

,rg_total,rg_curved
count,24.000,24.000
mean,83.750,0.709
std,33.695,0.084
min,15.000,0.538
25%,73.750,0.684
50%,87.500,0.719
75%,106.250,0.766
max,140.000,0.850


In [37]:
# Questions with mean score below 4 — flag for review
grade_cols = [f'rg_grade_{key}' for key in questions]
desc = df[grade_cols].describe()
low = desc.loc[:, desc.loc['mean'] < 4]

if low.empty:
    print('No questions with mean score below 4')
else:
    for col in low.columns:
        key = col[9:]
        print(f'Q{key}: mean={desc.loc["mean", col]:.1f}')
        print(f'  {configs[key]["question"]}')
        print()

low

Q17: mean=3.8
  In modeld, what's a 95% confidence interval for a player who is 27 years old, plays 1800 minutes per year, scores 11 points per game, and has a draft number=20?

Q31: mean=2.9
  Interpret the estimated coefficient on CPI in the model you have selected.



,rg_grade_17,rg_grade_31
count,24.000000,24.000000
mean,3.750000,2.916667
std,3.969996,4.402733
min,0.000000,0.000000
25%,0.000000,0.000000
50%,5.000000,0.000000
75%,5.000000,6.250000
max,10.000000,10.000000


## Save

In [38]:
output_path = 'data/Exam1_graded.xlsx'
df.to_excel(output_path, index=False)
print(f'Saved to {output_path}')
print(f'Students: {len(df)}')
print(f'Mean total: {df["rg_total"].mean():.1f}')
print(f'Mean curved: {df["rg_curved"].mean():.3f}')

Saved to data/Exam1_graded.xlsx
Students: 24
Mean total: 83.8
Mean curved: 0.709
